In [2]:
# =============================================================================
# Zelle 01 – Setup & finale Champion-Modelle (verifiziert per Code)
# =============================================================================
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import train_test_split

SEED = 42
Path("../models").mkdir(exist_ok=True)

# --- Modell B: Ridge, alpha=1.0 (Mehrheitswert, 4/5 Folds, code-verifiziert) ---
MODELL_B_FINAL = Ridge(alpha=1.0, random_state=SEED)

# --- Modell A: LogisticRegression, C=0.1, penalty=l2 (Mehrheitswert, 3/5 bzw. 4/5 Folds, code-verifiziert) ---
MODELL_A_FINAL = LogisticRegression(
    C=0.1, penalty="l2", solver="liblinear", class_weight="balanced",
    max_iter=1000, random_state=SEED
)

print("Modell B (Ridge) Parameter:")
print(MODELL_B_FINAL.get_params())
print("\nModell A (LogisticRegression) Parameter:")
print(MODELL_A_FINAL.get_params())

Modell B (Ridge) Parameter:
{'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': 42, 'solver': 'auto', 'tol': 0.0001}

Modell A (LogisticRegression) Parameter:
{'C': 0.1, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'liblinear', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


In [3]:
# =============================================================================
# Zelle 02 – Modell B (Ridge): Training auf vollem Trainingsset + Speicherung
# =============================================================================
from preprocessing import load_dataset_b, baue_preprocessing_pipeline_b, Y_B_MERKMALE

df_b = load_dataset_b("../data/processed/model_b_preprocessed.csv")

# Identischer Split wie in Notebook 11/12, fuer konsistente Test-Bewertung
train_idx_b, test_idx_b = train_test_split(df_b.index, test_size=0.2, random_state=SEED)

prep_b_final = baue_preprocessing_pipeline_b("original")
X_train_b = prep_b_final.fit_transform(df_b.loc[train_idx_b])
X_test_b = prep_b_final.transform(df_b.loc[test_idx_b])
y_train_b = df_b.loc[train_idx_b, Y_B_MERKMALE]
y_test_b = df_b.loc[test_idx_b, Y_B_MERKMALE]

MODELL_B_FINAL.fit(X_train_b, y_train_b)

from sklearn.metrics import r2_score
y_pred_test_b = MODELL_B_FINAL.predict(X_test_b)
test_r2_b = r2_score(y_test_b, y_pred_test_b)
print(f"Modell B finales Training abgeschlossen. Test-R² (Durchschnitt): {test_r2_b:.4f}")

# --- Speichern: Modell + Preprocessing-Pipeline gemeinsam (fuer konsistente Inferenz) ---
joblib.dump({"modell": MODELL_B_FINAL, "pipeline": prep_b_final, "y_spalten": Y_B_MERKMALE},
            "../models/modell_b_ridge_final.joblib")
print("Gespeichert in: ../models/modell_b_ridge_final.joblib")

Modell B finales Training abgeschlossen. Test-R² (Durchschnitt): 0.5912


PicklingError: Can't pickle <function baue_preprocessing_pipeline_b.<locals>.<lambda> at 0x000002B8B3BCB7E0>: it's not found as preprocessing.baue_preprocessing_pipeline_b.<locals>.<lambda>